 l'Étape 1 : La Collecte.   
 🎯 Objectif de cette étape :Parcourir le dossier docs/, lire tous les fichiers PDF et .txt rangés par service (RH/, IT/, General/), et extraire leur texte page par page en conservant le nom de la source, le numéro de page et le service d'origine

In [ ]:

from pathlib import Path
from pypdf import PdfReader

def lire_pdf(chemin):
    lecteur = PdfReader(chemin)
    pages = []
    for numero, page in enumerate(lecteur.pages, start=1):
        texte = page.extract_text()  # Remplacé par la méthode extract_text()
        pages.append({"source": chemin.name, "page": numero, "texte": texte})
    return pages

def lire_txt(chemin):
    # Lecture du fichier texte en UTF-8 et formatage en dictionnaire unique
    texte = chemin.read_text(encoding="utf-8")
    return [{"source": chemin.name, "page": 1, "texte": texte}]

def collecter(dossier="docs"):
    pages = []
    for chemin in sorted(Path(dossier).rglob("*")):
        if chemin.is_file():
            service = chemin.parent.name  # RH, IT ou General
            
            # Selon l'extension (.pdf ou .txt), on appelle la bonne fonction
            if chemin.suffix.lower() == ".pdf":
                fichiers_lus = lire_pdf(chemin)
            elif chemin.suffix.lower() == ".txt":
                fichiers_lus = lire_txt(chemin)
            else:
                continue
            
            # On ajoute la clé "service" à chaque page, puis on l'ajoute à la liste globale
            for p in fichiers_lus:
                p["service"] = service
                pages.append(p)
                
    return pages

if __name__ == "__main__":
    pages = collecter()
    print(len(pages), "pages lues")

23 pages lues


l'Étape 2 : Le Nettoyage

🎯 Objectif de cette étape :
Nettoyer le texte brut extrait des documents (supprimer les espaces multiples, normaliser les sauts de ligne excessifs et recoller les mots coupés en fin de ligne) pour le rendre propre avant le découpage.

In [ ]:

import re

def nettoyer(texte):
    # 1. Remplacer les espaces multiples ou tabulations par un seul espace
    texte = re.sub(r"[ \t]+", " ", texte)
    
    # 2. Remplacer 3 sauts de ligne ou plus par exactement 2 (pour garder les paragraphes)
    texte = re.sub(r"\n{3,}", "\n\n", texte)
    
    # 3. Recoller les mots coupés en fin de ligne (ex: "exem-\nple" -> "exemple")
    texte = re.sub(r"-\n(\w)", r"\1", texte)
    
    return texte.strip()

if __name__ == "__main__":
    # Test demandé par le prof
    exemple = "Ceci est   un    texte   avec   trop d'espaces.\n\n\nEt des sauts de ligne.\nEt un mot cou-\npé."
    print("--- AVANT ---")
    print(exemple)
    print("\n--- APRES ---")
    print(nettoyer(exemple))

--- AVANT ---
Ceci est   un    texte   avec   trop d'espaces.


Et des sauts de ligne.
Et un mot cou-
pé.

--- APRES ---
Ceci est un texte avec trop d'espaces.

Et des sauts de ligne.
Et un mot coupé.


l'Étape 3 : Le Découpage

🎯 Objectif de cette étape :
Découper les textes nettoyés en petits morceaux (chunks) de taille fixe avec un chevauchement (overlap), tout en générant un identifiant unique pour chaque morceau qui conserve sa source, sa page et son service.

In [ ]:

def decouper_texte(texte, taille_chunk=500, overlap=50):
    chunks = []
    i = 0
    while i < len(texte):
        fin = i + taille_chunk
        chunks.append(texte[i:fin])
        i += taille_chunk - overlap
    return chunks

def preparer_chunks():
    pages = collecter()  # Utilise la fonction collecter() définie plus haut dans le notebook
    tous_les_chunks = []
    
    for p in pages:
        texte_propre = nettoyer(p["texte"])  # Utilise la fonction nettoyer() définie plus haut
        morceaux = decouper_texte(texte_propre)
        
        for idx, chunk in enumerate(morceaux):
            tous_les_chunks.append({
                "id": f"{p['source']}_p{p['page']}_c{idx}",
                "texte": chunk,
                "source": p["source"],
                "page": p["page"],
                "service": p["service"]
            })
            
    return tous_les_chunks

# Exécution et test dans le notebook
chunks = preparer_chunks()
print(f"{len(chunks)} chunks générés au total.")

72 chunks générés au total.


l'Étape 4 : Vectorisation / Indexation

🎯Objectif de cette étape  :
Transformer chaque chunk de texte en vecteur (représentation mathématique) à l'aide d'un modèle d'embedding (via Ollama), puis stocker ces vecteurs dans une base de données vectorielle (ChromaDB) pour pouvoir faire des recherches par similarité plus tard.

In [11]:
from sentence_transformers import SentenceTransformer, util

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

phrases = [
    "Comment poser un jour de congé ?",
    "Quelle est la procédure pour demander des vacances ?",
    "Mon ordinateur ne se connecte pas au VPN.",
]

vecteurs = modele.encode(phrases)
print(vecteurs.shape)
print(util.cos_sim(vecteurs, vecteurs))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 690.06it/s]


(3, 384)
tensor([[ 1.0000,  0.4888, -0.0353],
        [ 0.4888,  1.0000, -0.0020],
        [-0.0353, -0.0020,  1.0000]])


In [12]:
import chromadb
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

client = chromadb.PersistentClient(path="base_vectorielle")
collection = client.get_or_create_collection(
    name="altea",
    metadata={"hnsw:space": "cosine"}
)

textes = [c["texte"] for c in chunks]  # Chunks issus de l'étape 3
vecteurs = modele.encode(textes, show_progress_bar=True).tolist()

collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=textes,
    embeddings=vecteurs,
    metadatas=[{"source": c["source"], "page": c["page"], "service": c["service"]} for c in chunks]
)

print(collection.count(), "chunks indexés")

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.08it/s]


72 chunks indexés


l'Étape 5 : La recherche par similarité - Retrieval
🎯 Objectif de cette étape : Interroger la base ChromaDB avec une question pour tester si elle retrouve automatiquement les bons documents pertinents.

In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
client = chromadb.PersistentClient(path="base_vectorielle")
collection = client.get_collection("altea")

def rechercher(question, k=3, service=None):
    vecteur = modele.encode([question]).tolist()[0]
    filtre = {"service": service} if service else None
    res = collection.query(
        query_embeddings=vecteur,
        n_results=k,
        where=filtre
    )
    resultats = []
    for texte, meta, distance in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        resultats.append({
            "texte": texte,
            "source": meta["source"],
            "page": meta["page"],
            "service": meta["service"],
            "score": 1 - distance  # Plus le score est proche de 1, plus le chunk est pertinent
        })
    return resultats

# Test de la recherche avec une question de salarié
rez = rechercher("Combien de jours de télétravail ?", k=3)
for i, r in enumerate(rez, start=1):
    print(f"--- Résultat {i} ---")
    print(f"Source : {r['source']} (Page {r['page']} - Service : {r['service']})")
    print(f"Score : {r['score']:.4f}")
    print(f"Extrait : {r['texte'][:150]}...\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1184.51it/s]


--- Résultat 1 ---
Source : teletravail.pdf (Page 1 - Service : RH)
Score : 0.6773
Extrait : bles.
Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de
leur tuteur.
2. Nombre de jours autorisés
Chaque sa...

--- Résultat 2 ---
Source : conges_absences.pdf (Page 1 - Service : RH)
Score : 0.6308
Extrait : de congés payés par mois de travail effectif, soit
30 jours ouvrables (5 semaines) pour une année complète. La période d'acquisition court du
1er juin...

--- Résultat 3 ---
Source : conges_absences.pdf (Page 1 - Service : RH)
Score : 0.5722
Extrait : oins 2 semaines à l'avance pour une absence de 1 à 3 jours, et
au moins 1 mois à l'avance pour une absence de plus d'une semaine....



l'Étape 6 : La génération de la réponse avec le LLM

🎯 Objectif de l'étape suivante : Maintenant que le système sait retrouver les bons morceaux de texte, on va les envoyer à un modèle de langage (LLM via Ollama) pour qu'il rédige une belle réponse claire et rédigée pour l'utilisateur.


In [18]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")
MODELE = "llama3.2"
REPLI = "Je ne trouve pas cette information dans la documentation."

def construire_prompt(question, extraits):
    contexte = ""
    for i, e in enumerate(extraits, start=1):
        contexte += f"[{i}] ({e['source']}, page {e['page']})\n{e['texte']}\n\n"
    
    prompt = f"""Tu es l'assistant interne d'Altéa Services.
Réponds à la question en utilisant UNIQUEMENT les extraits ci-dessous.
Après chaque information, indique le numéro de l'extrait entre crochets, par exemple [1].
Si les extraits ne contiennent pas la réponse, réponds exactement : "{REPLI}"

Extraits :
{contexte}

Question : {question}
Réponse :"""
    return prompt

def repondre(question, k=3):
    extraits = rechercher(question, k)
    prompt = construire_prompt(question, extraits)
    
    reponse = client.chat.completions.create(
        model=MODELE,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return reponse.choices[0].message.content, extraits

# Test de la génération
reponse_llm, extraits_utilises = repondre("Combien de jours de télétravail par semaine sont autorisés chez Altéa ?")
print("--- Réponse de l'Assistant ---")
print(reponse_llm)

--- Réponse de l'Assistant ---
2 jours par semaine, selon l'accord de l'employé et de son manager, avec un maximum de 2 jours par semaine autorisés pour chaque salarié éligible.


l'Étape 7 : Évaluation
🎯 Objectif de cette étape : Mesurer automatiquement si le système trouve les bons documents (via le taux de réussite pour $k=1, 3, 5$) pour garantir qu'il ne fournit pas de réponses au hasard.

In [19]:
import csv
import io

# 1. Simulation ou création du fichier evaluation.csv directement dans ton environnement notebook
csv_data = """question,document_attendu
Combien de jours de télétravail par semaine ?,teletravail.pdf
Comment réinitialiser mon mot de passe ?,mot_de_passe.txt"""

# Écriture dans un fichier temporaire pour le test
with open("evaluation.csv", "w", encoding="utf-8") as f:
    f.write(csv_data)

# 2. Fonction de calcul du taux de réussite demandée par le prof (adaptée au fichier unique)
def taux_reussite(fichier="evaluation.csv", k=3):
    with open(fichier, encoding="utf-8") as f:
        lignes = list(csv.DictReader(f))
        trouves = 0
        for ligne in lignes:
            # On utilise directement ta fonction 'rechercher' définie plus haut
            resultats_recherche = rechercher(ligne["question"], k)
            sources = [e["source"] for e in resultats_recherche]
            
            # Le document attendu fait-il partie des sources trouvées ?
            if ligne["document_attendu"] in sources:
                trouves += 1
        return (trouves / len(lignes)) * 100

# 3. Test pour k = 1, 3 et 5
for k in (1, 3, 5):
    taux = taux_reussite("evaluation.csv", k=k)
    print(f"k={k} : taux de réussite = {taux:.0f}% (où le bon document est trouvé)")

k=1 : taux de réussite = 100% (où le bon document est trouvé)
k=3 : taux de réussite = 100% (où le bon document est trouvé)
k=5 : taux de réussite = 100% (où le bon document est trouvé)


l'Étape 8- Mise à disposition et GitHub 

🎯 Objectif de cette étape : Offrir une interface graphique web (via Streamlit) pour que les utilisateurs finaux puissent interagir facilement avec l'assistant RAG, au lieu de passer par du code brut.